**Large Language Models**

_Copyright Notice_

This notebook is available for educational purposes. You may not use or distribute this notebook for commercial purposes. You may make copies of this notebook and use or distribute it for educational purposes as long as you cite the author as the source of the notebook.

Jaeyoung Chun | School of Applied Artificial Intelligence | Handong Global University

# Retrieval-Augmented Generation

## Load Environment Variables

In [1]:
from dotenv import load_dotenv

In [2]:
# retrieve API key and other environment variables
load_dotenv()

True

## Loading

In [3]:
from langchain_community.document_loaders import PyPDFLoader

In [4]:
loader = PyPDFLoader("./data/세계의-홍수-전설.pdf")

In [5]:
pages = loader.load_and_split()

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)


In [6]:
len(pages)

7

## Splitting

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

In [9]:
docs = text_splitter.split_documents(pages)

## Embeddings

In [10]:
from langchain_openai import OpenAIEmbeddings

In [11]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

## Vector Store

Persist in the `./chroma_db` directory

In [12]:
from langchain_chroma import Chroma

In [13]:
db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

## Retrieve

In [14]:
# MMR(Maximal Marginal Relevance)
retriever = db.as_retriever(
    search_type="mmr", search_kwargs={"k": 1, "fetch_k": 5, "lambda_mult": 0.8}
)

In [15]:
question = "노아의 홍수는 진짜 있었던 일인가요?"

In [16]:
context = retriever.invoke(question)

In [17]:
context

[Document(id='673eb1dc-459d-4cc3-8b67-d441da64cb57', metadata={'source': './data/세계의-홍수-전설.pdf', 'producer': 'macOS Version 14.5 (Build 23F79) Quartz PDFContext', 'moddate': "D:20241209114208Z00'00'", 'page': 6, 'page_label': '7', 'creator': 'Word', 'creationdate': "D:20241209114208Z00'00'", 'author': 'Jaeyoung Chun', 'total_pages': 7}, page_content="지금까지 이러한 각지에 남아있는 동서양의 홍수전설의 유사점들을 미루어볼 때, 그 \n근저에는 인류의 기억에 깊은 각인을 남긴 어떤 실제적 사건이 존재했었다고 생각할 수 \n있지 않을까?  \n결론적으로 말한다면 노아와 그의 가족에서부터 시작하여 바벨탑의 혼란에 의해 노아의 \n후손들이 지구 전역으로 퍼져나가게 되고, 각 대륙과 섬들에 정착한 노아의 후손들이 각기 \n뿌리를 내린 지역의 문화나 풍토성에 영향을 받아 창세기 홍수와 방주의 기본골격은 어느 \n정도 유지한 채 지역과 민족성에 따라 조상으로부터 대대로 내려왔던 이야기들이 조금씩 \n원전(原典)에서 광이 윤색되고 변질되는 형태로 구비전승(具備傳承)되면서, 종국적으로 \n원래 성서에 기록되어 있는 홍수이야기와는 상당히 동떨어진 홍수전설로 남게된 것으로 \n추정된다. 예를 들면 태평양의 피지섬에선 창세기에 나오는 직사각형의 방주(Ark) 형태가 \n피지섬 원주민들에게 쉽게 이해될 수 있고 생활에 유용하게 쓰이고 가장 친숙한 \n카누(Canoe)의 형태로 변색되어 나타나고 있다는 점을 들 수 있다. \n이러한 홍수전설의 분포도가 EXPO 창조과학전시관 노아 홍수관에 전시되어 있다. 그리고 \n전시관 옆으로 통하는 영화관에선 대홍수의 장면을 실감 있는 스크린으로 보여주기도 

## Augment

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

In [19]:
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="당신은 지적설계를 지지하는 과학자입니다."
        ),
        HumanMessagePromptTemplate.from_template(
            """
            {question}
            아래의 문맥에 기반하여 답해주세요.
            {context}
            """
        )
    ]
)

In [20]:
message = chat_template.format_messages(
    question=question,
    context=context[0].page_content
)

## Generate

In [21]:
from langchain_openai import ChatOpenAI

In [22]:
model = ChatOpenAI(
    model_name="gpt-6-luna",
)

In [23]:
for chunk in model.stream(message):
    print(chunk.content, end="", flush=True)

**짧게 답하면, 노아 홍수 이야기가 실제 홍수 기억을 바탕으로 했을 가능성은 있습니다. 하지만 세계 전체를 덮은 홍수가 실제로 일어났다는 사실이 홍수 전설의 유사성만으로 입증되지는 않습니다.**

지적설계를 지지하는 관점에서는 여러 민족의 홍수 전승을 인류가 겪은 큰 홍수의 오래된 기억으로 해석하고, 창세기의 노아 홍수와 연결해 볼 수 있습니다. 노아의 후손들이 흩어지면서 이야기가 각 지역의 환경과 문화에 맞게 달라졌다는 설명도 가능한 **신앙적·역사적 해석**입니다. 피지의 전승에서 방주가 카누처럼 묘사된다는 예도 그런 관점을 보여주는 사례로 제시됩니다.

다만 홍수 전승이 널리 퍼져 있다는 사실은 그 자체로 창세기의 사건, 노아의 가족, 바벨탑 이후의 인류 이동 경로를 입증하지는 않습니다. 큰 홍수는 여러 지역에서 반복해서 일어났고, 강이나 해안 주변에 살던 사람들이 그런 재난을 이야기로 전승했을 수도 있습니다. 서로 다른 전승의 공통점이 실제 사건의 기억인지, 문화 간 전파인지, 독립적으로 생겨난 이야기인지는 각각 검토해야 합니다.

또한 **지구 전체가 한 번의 홍수로 덮였다는 해석은 현재 주류 지질학의 증거와 맞지 않습니다.** 전 세계의 지층과 화석 기록은 하나의 최근 홍수로 설명되지 않으며, 지역마다 지질 형성과 환경 변화의 역사가 다르게 나타납니다. 반면 특정 지역에서 대규모 홍수가 일어났고 그것이 오래 기억되었을 가능성은 충분히 생각해 볼 수 있습니다.

따라서 제시된 문맥에 가장 신중하게 답하면 이렇습니다. **홍수 전승의 유사성은 인류가 큰 홍수들을 기억해 왔다는 점을 시사할 수 있지만, 그것만으로 성서에 기록된 전 지구적 노아 홍수가 과학적으로 확인되었다고 말할 수는 없습니다.** 노아 홍수를 실제 역사로 받아들이는 것은 신앙적 해석이며, 그 해석과 과학적 증거의 범위를 구분하는 것이 중요합니다.